# Импорты

In [ ]:
!git clone https://github.com/evagogua/denoiseML

In [ ]:
!pip install -r "/content/denoiseML/requirements.txt"

In [ ]:
import sys
import os

repo_name = "denoiseML"
os.chdir(f'/content/{repo_name}')

from src.models.denoise_model import (
    DenoiseDataset,
    make_last_subtoken_mask,
    prepare_denoise_dataset_from_json
)
from src.inference.denoiser import (
    denoising,
    predict_with_trainer,
    evaluate_on_test_set,
    test
)
from src.models.trainer import (
    plot_training_curves,
    compute_metrics,
    final_report
)

# Обучение

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-base")

### banking+credit_cards

In [ ]:
train_banking_credit_noise_dataset, test_banking_credit_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_banking_credit.json', tokenizer, test_size=0.2, seed=42, verbose=True)

train_travel_work_noise_dataset, test_travel_work_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_travel_work.json', tokenizer, test_size=0.2, seed=42, verbose=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, DataCollatorForTokenClassification
# from transformers.optimization import AdamW
from torch.optim import AdamW, Adam
import numpy as np

banking_credit_denoise_model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
optimizer = AdamW(banking_credit_denoise_model.parameters(), lr=1e-5, weight_decay=0.01)
training_args = TrainingArguments(
                  output_dir="trainer_logs", eval_strategy="steps", save_strategy='steps',
                  logging_strategy="steps", save_total_limit=2,
                  num_train_epochs=4, eval_steps=100, disable_tqdm=False,
                  metric_for_best_model='Accuracy',
                  warmup_ratio=0.1,
                  report_to="none"
              )
banking_credit_denoise_trainer = Trainer(
    model=banking_credit_denoise_model,
    optimizers=(optimizer, None),
    args=training_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    train_dataset=train_banking_credit_noise_dataset,
    eval_dataset=test_banking_credit_noise_dataset,
    compute_metrics=compute_metrics)
banking_credit_denoise_trainer.train()

In [ ]:
final_report(banking_credit_denoise_trainer, test_banking_credit_noise_dataset)

In [ ]:
final_report(banking_credit_denoise_trainer, test_travel_work_noise_dataset)

In [ ]:
plot_training_curves(banking_credit_denoise_trainer)

In [ ]:
evaluate_on_test_set(
    trainer=banking_credit_denoise_trainer,
    tokenizer=tokenizer,
    test_samples=test
)

In [ ]:
model_save_path = "banking_credit_denoise_model"
banking_credit_denoise_trainer.save_model(model_save_path)

### travel+work

In [ ]:
train_banking_credit_noise_dataset, test_banking_credit_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_banking_credit.json', tokenizer, test_size=0.2, seed=42, verbose=True)

train_travel_work_noise_dataset, test_travel_work_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_travel_work.json', tokenizer, test_size=0.2, seed=42, verbose=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, DataCollatorForTokenClassification
# from transformers.optimization import AdamW
from torch.optim import AdamW, Adam
import numpy as np

travel_work_denoise_model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
optimizer = AdamW(travel_work_denoise_model.parameters(), lr=1e-5, weight_decay=0.01)
training_args = TrainingArguments(
                  output_dir="trainer_logs", eval_strategy="steps", save_strategy='steps',
                  logging_strategy="steps", save_total_limit=2,
                  num_train_epochs=4, eval_steps=200, disable_tqdm=False,
                  metric_for_best_model='Accuracy',
                  warmup_ratio=0.1,
                  report_to="none"
              )
travel_work_denoise_trainer = Trainer(
    model=travel_work_denoise_model,
    optimizers=(optimizer, None),
    args=training_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    train_dataset=train_travel_work_noise_dataset,
    eval_dataset=test_travel_work_noise_dataset,
    compute_metrics=compute_metrics)
travel_work_denoise_trainer.train()

In [ ]:
plot_training_curves(travel_work_denoise_trainer)

In [ ]:
final_report(travel_work_denoise_trainer, test_travel_work_noise_dataset)

In [ ]:
final_report(travel_work_denoise_trainer, test_banking_credit_noise_dataset)

In [ ]:
evaluate_on_test_set(
    trainer=travel_work_denoise_trainer,
    tokenizer=tokenizer,
    test_samples=test,
)

In [ ]:
model_save_path = "travel_work_denoise_model"
travel_work_denoise_trainer.save_model(model_save_path)

### all_labels

In [ ]:
train_full_noise_dataset, test_full_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_all.json', tokenizer, test_size=0.2, seed=42, verbose=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, DataCollatorForTokenClassification
# from transformers.optimization import AdamW
from torch.optim import AdamW, Adam
import numpy as np

full_denoise_model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
optimizer = AdamW(full_denoise_model.parameters(), lr=1e-5, weight_decay=0.01)
training_args = TrainingArguments(
                  output_dir="trainer_logs", eval_strategy="steps", save_strategy='steps',
                  logging_strategy="steps", save_total_limit=2,
                  num_train_epochs=4, eval_steps=200, disable_tqdm=False,
                  metric_for_best_model='Accuracy',
                  warmup_ratio=0.1,
                  report_to="none"
              )
full_denoise_trainer = Trainer(
    model=full_denoise_model,
    optimizers=(optimizer, None),
    args=training_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    train_dataset=train_full_noise_dataset,
    eval_dataset=test_full_noise_dataset,
    compute_metrics=compute_metrics)
full_denoise_trainer.train()

In [ ]:
final_report(full_denoise_trainer, test_full_noise_dataset)

In [ ]:
plot_training_curves(full_denoise_trainer)

In [ ]:
evaluate_on_test_set(
    trainer=full_denoise_trainer,
    tokenizer=tokenizer,
    test_samples=test
)

In [ ]:
model_save_path = "full_denoise_model"
full_denoise_trainer.save_model(model_save_path)

### bctw_labels

In [ ]:
train_bctw_noise_dataset, test_bctw_noise_dataset = prepare_denoise_dataset_from_json('/content/denoiseML/data/noise_data_bctw.json', tokenizer, test_size=0.2, seed=42, verbose=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, DataCollatorForTokenClassification
# from transformers.optimization import AdamW
from torch.optim import AdamW, Adam
import numpy as np

bctw_denoise_model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
optimizer = AdamW(bctw_denoise_model.parameters(), lr=1e-5, weight_decay=0.01)
training_args = TrainingArguments(
                  output_dir="trainer_logs", eval_strategy="steps", save_strategy='steps',
                  logging_strategy="steps", save_total_limit=2,
                  num_train_epochs=4, eval_steps=200, disable_tqdm=False,
                  metric_for_best_model='Accuracy',
                  warmup_ratio=0.1,
                  report_to="none"
              )
bctw_denoise_trainer = Trainer(
    model=bctw_denoise_model,
    optimizers=(optimizer, None),
    args=training_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    train_dataset=train_bctw_noise_dataset,
    eval_dataset=test_bctw_noise_dataset,
    compute_metrics=compute_metrics)
bctw_denoise_trainer.train()

In [ ]:
final_report(bctw_denoise_trainer, test_bctw_noise_dataset)

In [ ]:
plot_training_curves(bctw_denoise_trainer)

In [ ]:
evaluate_on_test_set(
    trainer=bctw_denoise_trainer,
    tokenizer=tokenizer,
    test_samples=test
)

In [ ]:
model_save_path = "bctw_denoise_model"
bctw_denoise_trainer.save_model(model_save_path)